# 11. From Notebooks to picodrift

Notebooks 1–10 develop drifting on toy distributions. `picodrift/` applies it to real protein structure. It takes
[picofold](picofold), a 200-line model that generates the 16 Cα coordinates of a 16-residue protein fragment from its
sequence by flow matching, and replaces the flow matching with drifting. That repository assumes a handful of things
the tutorial never needed: what a protein fragment looks like as data, how structures are scored without aligning
them, why mirror images are a trap, how to build a chain from angles, what "physically valid" means, how picofold's
sampler works, and what happens to drifting when every input has exactly one reference output.

This notebook covers those prerequisites, each with the measurement that makes it concrete:

1. **Fragments as data.** The real training set, and the fact that makes the whole problem hard.
2. **Scoring without alignment.** lDDT and dRMSD, the floors they have on this data, and what they cannot see.
3. **Chirality.** Virtual torsions, why distance features are blind to mirror images, and the metric that is not.
4. **Internal coordinates.** Building a chain from angles, so bond lengths are exact by construction.
5. **Physical validity.** Clash and bond checks, with thresholds taken from the data.
6. **picofold's sampler.** Euler and Euler–Maruyama integration of a flow, and why few steps fail.
7. **One reference per input.** When the single-positive regime of notebook 8 hurts conditional drifting, and when it
   does not.

Read notebooks 3 (practical drifting), 5 (conditioning) and 8 (SE(3), Kabsch, the single positive) first. Then read
`picodrift/README.md`, which reports what happened at scale.

The data are picofold's own files (`picofold/train_data.txt`, `picofold/test_data.txt`). Every function below is a
compact copy of the one in `picodrift/picodrift.py` with the same name, re-defined here so the notebook stands alone.

In [ ]:
import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 110
device = "cuda" if torch.cuda.is_available() else "cpu"
DT = torch.float64            # geometry checks in float64; the toy trainings in §6-§7 drop to float32
L, AA = 16, "ACDEFGHIKLMNPQRSTVWY"
print("device:", device)

def load(path):
    """picofold's format: 'domain SEQUENCE x1 y1 z1 ... x16 y16 z16' per line -> ids, (N, L) ints, (N, L, 3) Å."""
    rows = [line.split() for line in open(path)]
    seq = torch.tensor([[AA.index(a) for a in r[1]] for r in rows])
    xyz = torch.tensor([[float(v) for v in r[2:]] for r in rows], dtype=DT).view(-1, L, 3)
    return [r[0] for r in rows], [r[1] for r in rows], seq, xyz - xyz.mean(1, keepdim=True)

train_ids, train_str, train_seq, train_xyz = load("picofold/train_data.txt")
test_ids, test_str, test_seq, test_xyz = load("picofold/test_data.txt")
print(f"train {len(train_ids)} windows from {len(set(train_ids))} domains; "
      f"test {len(test_ids)} windows from {len(set(test_ids))} domains; shared domains: {len(set(train_ids) & set(test_ids))}")

## 1. Protein fragments as data

A protein is a chain of amino acids (residues). Each residue has one Cα atom on the backbone, so a chain of 16
residues is, at this resolution, 16 points in 3-D. Each training example is a pair: a 16-letter sequence over the 20
amino acids, and the 16 Cα coordinates in Å. The windows were cut from CATH S40 domains, split by domain, so no test
window comes from a protein seen in training.

In [ ]:
from collections import Counter

bond = (train_xyz[:, 1:] - train_xyz[:, :-1]).norm(dim=-1).flatten()
rg = train_xyz.pow(2).sum(-1).mean(-1).sqrt()
dup = Counter(train_str)
print(f"consecutive Cα–Cα distance: median {bond.median():.2f} Å, 1st–99th pct {bond.quantile(0.01):.2f}–"
      f"{bond.quantile(0.99):.2f} Å, fraction < 3.2 Å (cis peptides) {(bond < 3.2).double().mean():.4f}")
print(f"radius of gyration: median {rg.median():.2f} Å (range {rg.min():.2f}–{rg.max():.2f})")
print(f"windows whose sequence occurs more than once: {sum(v for v in dup.values() if v > 1)} of {len(train_str)}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].hist(bond.numpy(), bins=200, range=(2.6, 4.3)); ax[0].set_yscale("log")
ax[0].set(xlabel="consecutive Cα–Cα distance (Å)", ylabel="count", title="(a) the chain is nearly rigid")
ax[1].hist(rg.numpy(), bins=80); ax[1].set(xlabel="radius of gyration (Å)", title="(b) helices are compact, strands extended")
plt.tight_layout(); plt.show()

The data show 46,859 training windows from 3,055 domains, 5,335 test windows from 336 domains, and no domain shared.
Three facts matter for everything that follows.

* **The chain is nearly rigid.** Consecutive Cα atoms sit 3.81 Å apart (1st–99th percentile 3.73–3.88 Å). The rare
  shorter bonds are *cis* peptides: 0.26% of bonds are under 3.2 Å. A generator that produces 2.5 or 5 Å steps is
  producing something that is not a protein, whatever its other scores say (§5).
* **Shapes vary a lot at fixed length.** The radius of gyration runs from 5.11 to 15.65 Å, median 7.80 Å. Helices are
  compact and strands extended. The distribution to be learned is multi-modal.
* **Each sequence has exactly one structure.** Only 47 of the 46,859 windows share a sequence with another window.
  This is the single-positive regime of notebook 8 §4, and it is what §7 below and most of `picodrift` are about.

## 2. Scoring a structure without aligning it

A generated structure comes out in an arbitrary orientation. Comparing raw coordinates would penalise a perfect
prediction that happens to face the other way. picofold's metrics compare *internal distances* instead: for each
residue pair, how far apart they are in the prediction versus the truth.

* **lDDT** (0–1, higher is better). Over residue pairs within 15 Å in the truth, it is the fraction of distance errors
  under 0.5, 1, 2 and 4 Å, averaged over the four thresholds. It is local and forgiving of a few bad pairs.
* **dRMSD** (Å, lower is better). The RMS error over all pairwise distances.

Calibrating them against known perturbations shows what the numbers mean on this data:

In [ ]:
def dmat(X):
    """(..., L, 3) -> (..., L, L), explicit differences so the diagonal is exactly 0."""
    return (X[..., :, None, :] - X[..., None, :, :]).pow(2).sum(-1).clamp_min(1e-24).sqrt()

UPPER = torch.ones(L, L, dtype=torch.bool).triu(1)

def lddt(pred, true, cutoff=15.0, thresholds=(0.5, 1.0, 2.0, 4.0)):
    """picofold's lDDT: pairs closer than `cutoff` in the truth, fraction of distance errors under each threshold."""
    dp, dt = dmat(pred), dmat(true)
    mask = (dt < cutoff) & UPPER
    err = (dp - dt).abs()
    passed = sum(((err < t) & mask).sum((-1, -2)) for t in thresholds)
    return passed.to(pred.dtype) / (len(thresholds) * mask.sum((-1, -2)))

def drmsd(pred, true):
    """RMS error over all pairwise distances (Å). No superposition needed."""
    return (dmat(pred)[..., UPPER] - dmat(true)[..., UPPER]).pow(2).mean(-1).sqrt()

g = torch.Generator().manual_seed(0)
true = test_xyz[:1000]
rand = train_xyz[torch.randint(0, len(train_xyz), (1000,), generator=g)]
print(f"{'prediction':38s} {'lDDT':>6s} {'dRMSD Å':>8s}")
print(f"{'the truth itself':38s} {lddt(true, true).mean():6.3f} {drmsd(true, true).mean():8.3f}")
for s in (0.5, 1.0, 2.0, 4.0):
    noisy = true + s * torch.randn(true.shape, generator=g, dtype=DT)
    print(f"{f'truth + {s} Å Gaussian noise per atom':38s} {lddt(noisy, true).mean():6.3f} {drmsd(noisy, true).mean():8.3f}")
print(f"{'a random real fragment':38s} {lddt(rand, true).mean():6.3f} {drmsd(rand, true).mean():8.3f}")
mirror = true * torch.tensor([1.0, 1.0, -1.0], dtype=DT)
print(f"{'the mirror image of the truth':38s} {lddt(mirror, true).mean():6.3f} {drmsd(mirror, true).mean():8.3f}")

Three calibrations to carry into `picodrift`:

* **lDDT falls fast with per-atom error.** 0.5 Å of noise per atom already gives 0.839, 1 Å gives 0.659, and 2 Å
  gives 0.440.
* **The floor is high.** A *random real fragment*, unrelated to the sequence, scores 0.514. Neighbouring residues are
  always 3.8 Å apart, and locally every fragment looks somewhat like every other. A model has learned something from
  the sequence only to the extent it beats this number. picodrift's naive drifting run (0.449) did not.
* **Both metrics are exactly blind to mirror images.** The mirror image scores lDDT 1.000 and dRMSD 0.000, because
  reflection preserves every distance. Real proteins are chiral: an α-helix is right-handed, and its mirror is not a
  protein. §3 builds the metric that can see this.

## 3. Chirality: what distances cannot see

Four consecutive Cα atoms define a *virtual torsion*: the dihedral angle about the middle bond. Three consecutive
atoms define a *virtual bond angle*. Bond angles are unchanged by reflection. Torsions **change sign**. In the IUPAC
sign convention a right-handed α-helix has a virtual torsion near +50°.

`mirror_frac` asks, per structure, whether the predicted torsions are closer to the truth's or to the truth's mirror
image. It is 0 for the truth, 1 for its mirror, and about 0.5 for something unrelated.

In [ ]:
def local_geometry(X):
    """cos of the L-2 virtual bond angles, and cos/sin of the L-3 virtual torsions (IUPAC sign)."""
    b = X[..., 1:, :] - X[..., :-1, :]
    bn = b / b.norm(dim=-1, keepdim=True)
    cos_angle = -(bn[..., :-1, :] * bn[..., 1:, :]).sum(-1)
    n1 = torch.cross(b[..., :-2, :], b[..., 1:-1, :], dim=-1)
    n2 = torch.cross(b[..., 1:-1, :], b[..., 2:, :], dim=-1)
    m = torch.cross(bn[..., 1:-1, :], n1, dim=-1)
    x, y = (n1 * n2).sum(-1), (m * n2).sum(-1)
    r = (x * x + y * y).sqrt()
    return cos_angle, x / r, y / r

def torsions(X):
    _, c, s = local_geometry(X)
    return torch.atan2(s, c)

def wrap(a):
    return torch.remainder(a + math.pi, 2 * math.pi) - math.pi

def mirror_frac(pred, true):
    """Fraction of predictions whose torsions are closer to the mirror image of the truth than to the truth."""
    tp, tt = torsions(pred), torsions(true)
    return (wrap(-tp - tt).abs().mean(-1) < wrap(tp - tt).abs().mean(-1)).double().mean().item()

tor = torsions(train_xyz[:20000]).flatten() * 180 / math.pi
helix = ((tor > 30) & (tor < 70)).double().mean()
print(f"virtual torsions in (30°, 70°), the right-handed helix: {helix:.3f}; "
      f"in (-70°, -30°), its mirror: {((tor > -70) & (tor < -30)).double().mean():.3f}")
print(f"mirror_frac(truth, truth) = {mirror_frac(true, true):.2f};  mirror_frac(mirror, truth) = {mirror_frac(mirror, true):.2f};"
      f"  random fragments: {mirror_frac(rand, true):.2f}")

pd = lambda X: dmat(X)[..., UPPER]
print(f"max |distance features(mirror) - distance features(truth)| = {(pd(mirror) - pd(true)).abs().max():.1e}")
_, c0, s0 = local_geometry(true); _, c1, s1 = local_geometry(mirror)
print(f"max |cos torsion difference| = {(c1 - c0).abs().max():.1e};   max |sin torsion + sin torsion(mirror)| = {(s1 + s0).abs().max():.1e}")

plt.figure(figsize=(6, 2.8))
plt.hist(tor.numpy(), bins=180, range=(-180, 180))
plt.xlabel("virtual Cα torsion (degrees)"); plt.ylabel("count"); plt.title("right-handed helices sit near +50°")
plt.tight_layout(); plt.show()

The histogram's dominant peak is the right-handed helix: 39.9% of torsions fall in (30°, 70°), against 2.1% in the
mirror window (−70°, −30°). The mirror metric behaves exactly as designed: 0.00 for the truth, 1.00 for its mirror,
0.49 for random fragments.

The last two lines are the reason picodrift has three feature sets rather than one. The 120 pairwise distances of a
structure and of its mirror are *identical*, to the last bit. A drifting field built on distances alone therefore
gives exactly zero signal about handedness. The cosines of the torsions are identical too, but the sines flip sign
exactly. picodrift's `local` feature (cos of the bond angles, cos and sin of the torsions) is the chiral one. The
`coord` feature is chiral too, because Kabsch alignment with the det = +1 guard (notebook 8 §2) refuses to reflect.

## 4. Internal coordinates: a chain that cannot break

§1 showed that real bonds are 3.8 Å. A generator that outputs 48 free Cartesian numbers has to *learn* that
constraint, and in `picodrift` it did not: naive drifting had a mean bond error of 1.1 Å. The alternative is to make
the constraint impossible to violate. The generator outputs angles and torsions, and the chain is built from them
with every bond exactly 3.8 Å. This is the "hard constraints belong in the parameterisation" lesson of notebook 8 §6.

`chain_from_internal` is the NeRF construction. It places three atoms, then each next atom at the given bond length,
bond angle and torsion relative to the previous three. To trust it, it must be the exact inverse of `local_geometry`:

In [ ]:
def chain_from_internal(theta, phi, bond=3.8):
    """Inverse of local_geometry: interior angles theta (..., L-2) and torsions phi (..., L-3) -> (..., L, 3),
    every consecutive distance exactly `bond` (NeRF placement)."""
    z = theta.new_zeros(theta.shape[:-1])
    pts = [torch.stack([z, z, z], -1), torch.stack([z + bond, z, z], -1)]
    pts.append(pts[1] + bond * torch.stack([-torch.cos(theta[..., 0]), torch.sin(theta[..., 0]), z], -1))
    for k in range(phi.shape[-1]):
        a, b, c = pts[-3], pts[-2], pts[-1]
        bc = (c - b) / (c - b).norm(dim=-1, keepdim=True)
        n = torch.cross(b - a, bc, dim=-1); n = n / n.norm(dim=-1, keepdim=True)
        m = torch.cross(n, bc, dim=-1)
        th, ph = theta[..., k + 1], phi[..., k]
        pts.append(c + (-bond * torch.cos(th))[..., None] * bc + (bond * torch.sin(th) * torch.cos(ph))[..., None] * m
                   + (bond * torch.sin(th) * torch.sin(ph))[..., None] * n)
    X = torch.stack(pts, -2)
    return X - X.mean(-2, keepdim=True)

def kabsch(P, Q):
    """Rotation R with P @ R ~= Q for centred inputs; det(R) = +1 (notebook 8 §2)."""
    U, _, Vh = torch.linalg.svd(P.transpose(-1, -2) @ Q)
    d = torch.sign(torch.linalg.det(Vh.transpose(-1, -2) @ U.transpose(-1, -2)))
    D = torch.eye(3, dtype=P.dtype).expand(*P.shape[:-2], 3, 3).clone(); D[..., 2, 2] = d
    return U @ D @ Vh

def rmsd(P, Q):
    return (P @ kabsch(P, Q) - Q).pow(2).sum(-1).mean(-1).sqrt()

ca, c, s = local_geometry(true)
theta, phi = torch.acos(ca), torch.atan2(s, c)
for name, ph in (("same torsion sign", phi), ("torsion sign flipped", -phi)):
    R = chain_from_internal(theta, ph)
    print(f"{name:22s}: max torsion error {wrap(torsions(R) - ph).abs().max():.1e} rad | RMSD to truth median "
          f"{rmsd(R, true).median():.3f} Å | lDDT {lddt(R, true).mean():.4f} | mirror_frac {mirror_frac(R, true):.2f}")

The round trip is exact: the torsion error is $2.7\times10^{-15}$ rad, so the builder and the feature extractor agree
on the sign convention. Fixing every bond at 3.8 Å moves real fragments by a median 0.034 Å and costs only 0.002 lDDT
(0.9984), the price of ignoring the rare *cis* bonds.

The second line is a warning worth remembering. Flip the torsion sign convention and the rebuilt chain is the
**mirror image**: 3.176 Å RMSD from the truth, `mirror_frac` 1.00, and an identical lDDT of 0.9984. A sign error
between the generator's output head and the feature code would therefore train a model that looks perfect on lDDT
while producing left-handed helices. Only the torsion check in `picodrift`'s `selftest` would catch it. In picodrift's
round 2, an internal-coordinate *regressor* scored a respectable lDDT while 61% of its samples were mirror images.

## 5. Physical validity: bonds and clashes

Two residues that are not neighbours on the chain cannot sit on top of each other. Rather than pick a clash threshold
from a textbook, measure it:

In [ ]:
sep = (torch.arange(L)[:, None] - torch.arange(L)[None]).abs()
D = dmat(train_xyz)
for lo, hi, lab in ((1, 1, "|i-j| = 1"), (2, 2, "|i-j| = 2"), (3, L, "|i-j| >= 3")):
    v = D[:, (sep >= lo) & (sep <= hi)].flatten()
    v = v[torch.randperm(len(v), generator=g)[:2_000_000]]
    print(f"{lab:10s}: min {v.min():.2f} Å, 0.1th percentile {v.quantile(0.001):.2f} Å, median {v.median():.2f} Å")

def clash_frac(X, cut=3.5):
    return ((dmat(X) < cut) & (sep >= 2)).flatten(-2).any(-1).double().mean().item()

def bond_err(X, ref):
    return ((X[..., 1:, :] - X[..., :-1, :]).norm(dim=-1) - (ref[..., 1:, :] - ref[..., :-1, :]).norm(dim=-1)).abs().mean().item()

blob = torch.randn(true.shape, generator=g, dtype=DT)
blob = blob - blob.mean(1, keepdim=True)
blob = blob * (true.pow(2).sum(-1).mean(-1).sqrt() / blob.pow(2).sum(-1).mean(-1).sqrt())[:, None, None]
print(f"\n{'':32s} {'clash':>6s} {'bond err Å':>10s} {'lDDT':>6s}")
for name, X in (("real fragments", true), ("random real fragment", rand),
                ("Gaussian blob, matched Rg", blob), ("mean of 200 aligned fragments", None)):
    if X is None:  # the conditional mean of an ambiguous set: align 200 fragments onto one, average
        A = train_xyz[:200]; X = (A @ kabsch(A, A[:1].expand_as(A))).mean(0, keepdim=True).expand_as(true)
    print(f"{name:32s} {clash_frac(X):6.3f} {bond_err(X, true):10.3f} {lddt(X, true).mean():6.3f}")

No real pair two or more residues apart comes closer than 2.89 Å, and 99.9% of $|i-j|\ge3$ pairs are farther than
4.29 Å. picodrift therefore flags a structure as clashing if any $|i-j|\ge2$ pair is under **3.5 Å**. That flags 0.2%
of these real test fragments: rare, but not never, which is why the threshold is not 4 Å.

The table shows why validity is reported next to lDDT. A Gaussian blob with the right radius of gyration clashes in
91.4% of cases, has a 7.67 Å bond error, and scores lDDT 0.221. The more instructive row is the **average of 200
aligned real fragments**: the conditional mean of an ambiguous set. It clashes every time (1.000), its bonds are 2.0 Å
off, and its lDDT of 0.406 is still not catastrophic. Averaging over possibilities produces something that is
plausible in shape and impossible as a protein. That is exactly what a regression model, or a drifting model that has
lost its positives, will output.

## 6. picofold's sampler, and why steps cost quality

Flow matching (notebook 6) learns a velocity $v_\theta(x_t, t)$ along the straight path
$x_t = t\,y + (1-t)\,\varepsilon$ and integrates it from noise at $t=0$ to data at $t=1$. picofold uses two samplers:

* **Euler**: $x \leftarrow x + v\,\Delta t$, the probability-flow ODE.
* **Euler–Maruyama (EM)**: adds a score correction and fresh noise,
  $x \leftarrow x + (v + w\,s)\,\Delta t + \sqrt{2\,\Delta t\,w\,\tau}\;\xi$, with $s = -(x - t v)/(1-t)$,
  $w = (1-t)/(t+0.01)$ and $\tau = 0.3$.

The number of steps is the number of network evaluations (NFE), and it is the cost that drifting is designed to
remove. On a 2-D ring of eight Gaussians, picofold's training objective and picofold's sampler give the following
total-variation distances to the data (lower is better):

In [ ]:
# A 2-D stand-in for the data: eight Gaussians on a ring.
def ring(n, g=None):
    k = torch.randint(0, 8, (n,), generator=g)
    a = k * (2 * math.pi / 8)
    return torch.stack([a.cos(), a.sin()], -1) * 2 + 0.1 * torch.randn(n, 2, generator=g)

def tv2d(A, B, bins=40, lim=3.0):
    h = lambda z: (lambda H: H / H.sum())(torch.histogramdd(z.float().cpu(), bins=[bins, bins], range=[-lim, lim, -lim, lim])[0])
    return 0.5 * (h(A) - h(B)).abs().sum().item()

class VelNet(nn.Module):
    def __init__(self, h=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, 2))
    def forward(self, x, t):
        return self.net(torch.cat([x, t[:, None]], -1))

torch.manual_seed(0)
vel = VelNet().to(device)
opt = torch.optim.Adam(vel.parameters(), 1e-3)
for step in range(4000):                                  # picofold's objective, unchanged
    x1 = ring(512).to(device)
    t = torch.sigmoid(torch.randn(512, device=device) * 1.7 + 0.8)
    x0 = torch.randn_like(x1)
    loss = F.mse_loss(vel(t[:, None] * x1 + (1 - t[:, None]) * x0, t), x1 - x0)
    opt.zero_grad(); loss.backward(); opt.step()

@torch.no_grad()
def sample(n, steps, sampler, tau=0.3):                   # picofold's generate(), in 2-D
    x = torch.randn(n, 2, device=device); dt = 1.0 / steps
    for i in range(steps):
        t = i * dt
        v = vel(x, torch.full((n,), t, device=device))
        if sampler == "em" and t < 0.99:
            score = -(x - t * v) / (1 - t); w = (1 - t) / (t + 0.01)
            x = x + (v + w * score) * dt + math.sqrt(2 * dt * w * tau) * torch.randn_like(x)
        else:
            x = x + v * dt
    return x

ref = ring(20000)
print(f"TV floor (two independent data draws): {tv2d(ring(20000), ref):.3f}")
print(f"{'NFE':>4s} {'Euler':>7s} {'Euler-Maruyama':>15s}")
res = {}
for nfe in (1, 2, 5, 10, 20, 50, 200):
    res[nfe] = (tv2d(sample(20000, nfe, "euler"), ref), tv2d(sample(20000, nfe, "em"), ref))
    print(f"{nfe:4d} {res[nfe][0]:7.3f} {res[nfe][1]:15.3f}")
print(f"\nEM correction weight w(t) = (1-t)/(t+0.01) at t = 0: {1 / 0.01:.0f}; a single step of size dt = 1 multiplies "
      f"the score term by {1 / 0.01:.0f}")

Two lessons carry over.

* **Few steps fail, for a structural reason.** At 1–2 NFE both samplers are useless (TV 0.94–1.00). EM is useless in
  a particular way: its correction weight $w(0) = 100$, so one step of size $\Delta t = 1$ multiplies the score term
  by 100 and throws the samples far off. On fragments the same thing happens: picodrift measured lDDT 0.000 at 1 EM
  step, with dRMSDs in the thousands of Å.
* **Which sampler wins is data-dependent.** On this ring, plain Euler is better at most step counts (0.186 at 200
  steps against EM's 0.364). On protein fragments the opposite holds: picodrift reproduced picofold's EM at lDDT 0.621
  against Euler's 0.558. EM's correction-plus-noise is a tuned heuristic, not a free improvement. This is why
  picodrift's comparison sweeps both samplers at every NFE rather than quoting one.

The cost axis is the point of drifting. A one-step drifting generator is a single forward pass, and the question
`picodrift` answers is how much quality that costs against a 200-step sampler.

## 7. One reference per input: when it hurts

Notebook 8 §4 showed that with **one** input and **one** reference, the fixed point of drifting is the reference
itself: the ensemble collapses. A conditional model sees thousands of inputs, each with one reference, and a single
network shared between them. What happens then depends on how much the input says about the output.

The toy below has a condition $c\in[0,1]$ standing in for the sequence, and an output on an 8-mode ring standing in
for the structure. Each of 4096 training conditions has **one** observed output. Two regimes:

* **informative**: $c$ identifies the mode, like a sequence that determines its structure;
* **uninformative**: $c$ says nothing about which mode, like sequences whose structure the network cannot predict.

The generator is trained with the batched, jointly normalised field of notebook 3, with one of two positive sets per
condition: the **single** observed output, or a **set** of 16 fresh draws from the true conditional. The set stands in
for samples from a teacher model, which is what picodrift's "flow bank" provides. A fifth run trains on only 16
conditions, few enough for the network to memorise each one. The condition enters the generator through Fourier
features, so a sharp condition-to-mode map is representable. The network is identical in every regime.

In [ ]:
# The conditional toy. A condition c in [0, 1] plays the sequence; y | c lives on an 8-mode ring (radius 2). Each
# training condition has exactly ONE observed y, as each sequence has one reference structure. Two regimes:
#   informative:   c identifies the mode (c lies inside bin k of 8 equal bins, away from the bin edges, and the mode
#                  is k), like a sequence that determines its structure;
#   uninformative: c says nothing about which mode, like sequences whose structure the network cannot predict.
M = 8

def draw_c(n, informative, g=None):
    """Conditions. Informative ones sit in the middle 60% of their bin, so 'which mode' is a learnable function of c
    rather than a step a smooth network must straddle."""
    if not informative:
        return torch.rand(n, generator=g)
    k = torch.randint(0, M, (n,), generator=g)
    return (k + 0.5 + 0.6 * (torch.rand(n, generator=g) - 0.5)) / M

def sample_y(c, informative, g=None):
    k = (c * M).long().clamp(max=M - 1) if informative else torch.randint(0, M, c.shape, generator=g)
    a = k * (2 * math.pi / M)
    return 2 * torch.stack([a.cos(), a.sin()], -1) + 0.1 * torch.randn(*c.shape, 2, generator=g)

def compute_V(x, y_pos, y_neg=None, tau=0.2):
    """Notebook 3's field, batched over a leading 'input' axis: x (B, K, C), y_pos (B, P, C).
    y_neg=None means the samples themselves, diagonal masked by -1e6. A = sqrt(softmax_row * softmax_col)."""
    K = x.shape[1]
    if y_neg is None:
        y_neg = x
        ln = -torch.cdist(x, x) / tau - torch.eye(K, device=x.device) * 1e6
    else:
        ln = -torch.cdist(x, y_neg) / tau
    logit = torch.cat([-torch.cdist(x, y_pos) / tau, ln], -1)
    A = torch.sqrt(logit.softmax(-1) * logit.softmax(-2))
    P = y_pos.shape[1]
    Ap, An = A[..., :P], A[..., P:]
    return (Ap * An.sum(-1, keepdim=True)) @ y_pos - (An * Ap.sum(-1, keepdim=True)) @ y_neg

class CondGen(nn.Module):
    """Generator f(eps, c). c enters through Fourier features (frequencies 1, 2, 4, 8), so a sharp c -> mode map is
    representable; the same network is used in every regime."""
    FREQS = torch.tensor([1.0, 2.0, 4.0, 8.0])
    def __init__(self, h=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2 + 2 * len(self.FREQS), h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(),
                                 nn.Linear(h, h), nn.SiLU(), nn.Linear(h, 2))
    def forward(self, eps, c):
        a = 2 * math.pi * c[..., None] * self.FREQS.to(c.device)
        return self.net(torch.cat([eps, a.sin(), a.cos()], -1))

def train_conditional(informative, positives, n_cond=4096, steps=2500, B=64, K=16, P=16,
                      taus=(0.05, 0.2, 0.8), eta=0.05, seed=0):
    """positives: 'single' (the one observed y per condition) or 'set' (P fresh draws from the true conditional,
    standing in for a teacher's samples)."""
    gg = torch.Generator().manual_seed(seed); torch.manual_seed(seed)
    C = draw_c(n_cond, informative, gg); Yobs = sample_y(C, informative, gg)
    gen = CondGen().to(device)
    opt = torch.optim.Adam(gen.parameters(), 2e-3, betas=(0.9, 0.95))
    for step in range(steps):
        i = torch.randint(0, n_cond, (B,), generator=gg)
        x = gen(torch.randn(B, K, 2, device=device), C[i][:, None].expand(B, K).to(device))
        yp = (Yobs[i][:, None] if positives == "single"
              else sample_y(C[i][:, None].expand(B, P), informative, gg)).to(device)
        with torch.no_grad():   # per-tau drift normalisation before summing (notebook 3)
            V = sum((lambda v: v / v.pow(2).mean().sqrt().clamp_min(1e-8))(compute_V(x, yp, None, t * math.sqrt(2)))
                    for t in taus)
        loss = F.mse_loss(x, (x + eta * V).detach())
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(gen.parameters(), 2.0); opt.step()
    return gen, C, Yobs

def ring_stats(x):
    """Distance from the nearest mode (Å-like units; the 0.1 data noise sets the floor) and the radial spread."""
    a = torch.atan2(x[..., 1], x[..., 0]); r = x.norm(dim=-1)
    m = torch.round(a / (2 * math.pi / M)) * (2 * math.pi / M)
    near = (x - 2 * torch.stack([m.cos(), m.sin()], -1)).norm(dim=-1)
    return near.mean().item(), r.std().item()

@torch.no_grad()
def draw(gen, c, n=100):
    return gen(torch.randn(len(c), n, 2, device=device), c[:, None].expand(-1, n).to(device)).cpu()

rows, samples = [], {}
t0 = time.time()
for informative in (True, False):
    c_eval = draw_c(200, informative, g)
    truth = sample_y(c_eval[:, None].expand(-1, 100), informative, g)
    rows.append((informative, "true conditional", *ring_stats(truth), truth.std(1).mean().item()))
    for positives in ("single", "set"):
        gen, _, _ = train_conditional(informative, positives)
        x = draw(gen, c_eval); samples[(informative, positives)] = x
        rows.append((informative, f"{positives} positive{'s' if positives == 'set' else ''}", *ring_stats(x),
                     x.std(1).mean().item()))
gen, Cm, Ym = train_conditional(False, "single", n_cond=16, steps=5000)   # few enough conditions to memorise
xm = draw(gen, Cm)
rows.append((False, "single, 16 conditions", *ring_stats(xm), xm.std(1).mean().item()))
memo_err = (xm - Ym[:, None]).norm(dim=-1).mean().item()
print(f"trained five generators in {time.time() - t0:.0f} s\n")
print(f"{'condition':14s} {'positives':24s} {'dist to nearest mode':>21s} {'radial std':>11s} {'std within c':>13s}")
for inf, name, near, rstd, wstd in rows:
    print(f"{'informative' if inf else 'uninformative':14s} {name:24s} {near:21.3f} {rstd:11.3f} {wstd:13.3f}")
print(f"\nmemorised run: mean distance from each condition's samples to its one observed y = {memo_err:.3f}")

In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(17, 3.6))
panels = [(f"{'informative' if i else 'uninformative'}, {p}", samples[(i, p)]) for i in (True, False) for p in ("single", "set")]
panels.append(("uninformative, single, 16 conditions", xm))
for a, (name, x) in zip(ax, panels):
    a.scatter(x[..., 0].flatten(), x[..., 1].flatten(), s=0.5, alpha=0.4)
    a.set_title(name, fontsize=9); a.set_xlim(-3, 3); a.set_ylim(-3, 3); a.set_aspect("equal")
plt.tight_layout(); plt.show()

Read the "dist to nearest mode" column against the truth's 0.125, the floor set by the 0.1 data noise.

* **With an informative condition, one positive is nearly enough.** The single-positive generator lands 0.164 from the
  modes, the positive-set generator 0.128, and both put clean, separate blobs on the eight modes. The joint
  row-and-column normalisation is why one positive works. The column softmax runs over the $K$ samples, so the lone
  positive's attraction goes mostly to the samples nearest to it. The attraction stays local even though
  $\bar k \equiv 1$ in the naive form (`CLAUDE.md` pitfall 5).
* **With an uninformative condition, one positive costs a lot.** The positive set gets samples onto the modes (0.207,
  radial std 0.171). The single positive smears them between and around the modes (0.327, radial std 0.264), and the
  figure shows the mass spilling into the interior of the ring. Each condition sees one mode per step, and the shared
  network has to reconcile conditions it cannot tell apart. The result has the right overall spread (within-condition
  std 1.429 against the truth's 1.416) but puts the mass in the wrong places.
* **Memorisable conditions collapse.** With only 16 conditions, each condition's samples shrink to a within-condition
  std of 0.019, against the true conditional's 1.416, and sit 0.036 from their one observed point. This is notebook
  8's collapse, appearing as soon as the network can separate its inputs.

Protein fragments are in the uninformative regime. In `picodrift`, the structures of a test sequence's BLOSUM62-nearest
training neighbours score lDDT 0.563, barely above random fragments (0.522): sequence says little about local
structure that a nearest-neighbour lookup can use. Naive drifting there produced this toy's failure at much larger
scale. Its ensembles were over-dispersed (4.5 Å spread against flow matching's 2.4 Å) and non-protein-like, until
positive *sets* sampled from a trained flow model replaced the single references. The toy shows the direction of the
effect, not its size. picodrift's features live in 48–120 dimensions, not 2, and its failure was far more severe than
the 0.327-vs-0.207 gap here.

The memorisable run is picodrift's round-2 "memorisation test" in miniature. On 512 sequences, picodrift's field
collapsed each ensemble onto its reference (training lDDT 0.994), confirming that the field itself was not broken.

## 8. Where each idea lives in picodrift

| idea here | picodrift (`picodrift/picodrift.py`) | where it matters (`picodrift/README.md`) |
|---|---|---|
| fragment data, one structure per sequence (§1) | `load_windows`, `Data` | §3, §5 |
| lDDT, dRMSD, the random-fragment floor (§2) | `lddt`, `drmsd`, `evaluate_run` baselines | §4.1 |
| torsions, `mirror_frac`, chiral and achiral features (§3) | `local_geometry`, `structure_metrics`, `featurize` | §2, §4.5 |
| internal coordinates and the sign trap (§4) | `chain_from_internal`, `--output internal`, `selftest` | §4.5 |
| clash threshold and bond error (§5) | `CLASH_DIST`, `clash_penalty`, `--clash-weight` | §4.1, §4.5 |
| Euler vs EM and the NFE axis (§6) | `FlowMatching.sample`, `eval_nfe` | §4.2 |
| the single-positive regime and positive sets (§7) | `Drifting.positive_set`, `--positives`, `distill` | §4.5–4.7, §5 |

## Summary

* Fragments are 16 Cα points with nearly rigid 3.81 Å bonds, very variable shape, and **one structure per sequence**
  (47 of 46,859 windows share a sequence).
* lDDT has a high floor on this data (a random real fragment scores 0.514) and, like dRMSD, is **exactly blind to
  mirror images**. Virtual torsions are not, and `mirror_frac` measures chirality directly.
* Pairwise-distance features cannot see handedness: they are bit-identical for a structure and its mirror. Torsion
  sines flip sign, which makes `local` the chiral feature.
* Building chains from internal coordinates makes bonds exact by construction, at a cost of 0.002 lDDT. A
  sign-convention error yields the mirror image with *identical* lDDT.
* A 3.5 Å clash threshold flags 0.2% of real fragments. Averaging aligned fragments gives a plausible shape that
  clashes 100% of the time.
* Flow samplers fail at 1–2 steps (EM's $w(0)=100$), and which sampler wins depends on the data.
* **One reference per input is nearly harmless when the input determines the output, and costly when it does not.** It
  collapses when the network can memorise. Protein fragments sit in the costly regime, which is why picodrift needed
  positive sets distilled from a teacher.

---

Next: [`picodrift/README.md`](picodrift/README.md) runs all of this at scale on Modal, and
[`picodrift/reports/picodrift_report.md`](picodrift/reports/picodrift_report.md) is the full write-up.

Optional extension: [12. Looped Drifting](<12. Looped Drifting.ipynb>) asks what a test-time depth knob, a verifier and a filter would do for a one-step model.